In [2]:
import pandas as pd
import json 
import io
import zstandard as zstd
from tqdm import tqdm
import os

In [3]:
df=pd.read_csv("data/subreddits/dataframe_21.csv")
df.head()

C:\Users\elisa\AppData\Local\Temp\ipykernel_39108\584388149.py:1: DtypeWarning: Columns (0: advertiser_category, 1: banner_background_color, 2: banner_background_image, 3: community_icon, 4: description, 5: emojis_custom_size, 6: header_img, 7: header_size, 8: header_title, 9: is_crosspostable_subreddit, 10: is_default_banner, 11: is_default_icon, 12: key_color, 13: link_flair_position, 14: mobile_banner_image, 15: primary_color, 16: submit_link_label, 17: submit_text, 18: submit_text_html, 19: submit_text_label, 20: wiki_enabled, 21: interstitial_warning_message, 22: quarantine_message, 23: quarantine_message_html, 24: quarantine_permissions, 25: content_category) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("data/subreddits/dataframe_21.csv")


,_meta,accept_followers,accounts_active,accounts_active_is_fuzzed,active_user_count,advertiser_category,all_original_content,allow_discovery,allow_galleries,allow_images,...,user_sr_theme_enabled,videostream_links_count,wiki_enabled,wls,whitelist_status,interstitial_warning_message,quarantine_message,quarantine_message_html,quarantine_permissions,content_category
0,"{'earliest_comment_at': None, 'earliest_post_a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"{'earliest_comment_at': None, 'earliest_post_a...",True,NaN,False,NaN,NaN,False,True,True,True,...,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"{'earliest_comment_at': None, 'earliest_post_a...",True,NaN,False,NaN,NaN,False,True,True,True,...,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"{'earliest_comment_at': None, 'earliest_post_a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"{'earliest_comment_at': 1709266657, 'earliest_...",True,NaN,False,NaN,NaN,False,True,True,True,...,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.columns.tolist()

['_meta',
 'accept_followers',
 'accounts_active',
 'accounts_active_is_fuzzed',
 'active_user_count',
 'advertiser_category',
 'all_original_content',
 'allow_discovery',
 'allow_galleries',
 'allow_images',
 'allow_polls',
 'allow_prediction_contributors',
 'allow_predictions',
 'allow_predictions_tournament',
 'allow_talks',
 'allow_videogifs',
 'allow_videos',
 'allowed_media_in_comments',
 'banner_background_color',
 'banner_background_image',
 'banner_img',
 'banner_size',
 'can_assign_link_flair',
 'can_assign_user_flair',
 'collapse_deleted_comments',
 'comment_contribution_settings',
 'comment_score_hide_mins',
 'community_icon',
 'community_reviewed',
 'created',
 'created_utc',
 'description',
 'disable_contributor_requests',
 'display_name',
 'display_name_prefixed',
 'emojis_custom_size',
 'emojis_enabled',
 'free_form_reports',
 'has_menu_widget',
 'header_img',
 'header_size',
 'header_title',
 'hide_ads',
 'icon_img',
 'icon_size',
 'id',
 'is_crosspostable_subreddit',


In [6]:
", ".join(df.columns.tolist())

'_meta, accept_followers, accounts_active, accounts_active_is_fuzzed, active_user_count, advertiser_category, all_original_content, allow_discovery, allow_galleries, allow_images, allow_polls, allow_prediction_contributors, allow_predictions, allow_predictions_tournament, allow_talks, allow_videogifs, allow_videos, allowed_media_in_comments, banner_background_color, banner_background_image, banner_img, banner_size, can_assign_link_flair, can_assign_user_flair, collapse_deleted_comments, comment_contribution_settings, comment_score_hide_mins, community_icon, community_reviewed, created, created_utc, description, disable_contributor_requests, display_name, display_name_prefixed, emojis_custom_size, emojis_enabled, free_form_reports, has_menu_widget, header_img, header_size, header_title, hide_ads, icon_img, icon_size, id, is_crosspostable_subreddit, is_default_banner, is_default_icon, is_enrolled_in_new_modmail, key_color, lang, link_flair_enabled, link_flair_position, mobile_banner_imag

In [6]:
filepath = "./reddit/subreddits/subreddits_2025-01.zst"
print(os.path.getsize(filepath) / 1e6, "MB")

2240.562367 MB


### Function for loading zst files

In [ ]:

#Bruger chunks så den ikke crasher pga memory overload
def load_zst_json(filepath,year, chunk_size=10000):
    chunks = []
    buffer = []


    with open(filepath, "rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            text_stream = io.TextIOWrapper(reader, encoding="utf-8")

            num_dataframes=0
            for i, line in tqdm(enumerate(text_stream), desc= "Amount of subreddits processed"):
                buffer.append(json.loads(line)) #Converts each json line to python dictionary

                if len(buffer) >= chunk_size:
                    chunks.append(pd.DataFrame(buffer))
                    buffer = []
                    #Tilføj denne linje hvis du kun vil have et subset ud. 100 svarer til 100000 subreddits
                    #if len(chunks) >= 100: return pd.concat(chunks, ignore_index=True)

                    # save file for each 1 million subreddits
                    if len(chunks) >= 100:
                        df=pd.concat(chunks,ignore_index=True)
                        df.to_csv(f"./data/subreddits/{year}_dataframe_{num_dataframes}.csv", index=False)
                        # reset chunks and buffer
                        chunks = []
                        buffer = []
                        num_dataframes+=1

            # remaining rows
            if buffer:
                chunks.append(pd.DataFrame(buffer))

    # save the remaining chunks to csv         
    if len(chunks) > 0:
        df = pd.concat(chunks, ignore_index=True)
        df.to_csv(f"./data/subreddits/{year}_dataframe_{num_dataframes}.csv", index=False)

### Loading the data into dataframes

In [9]:
load_zst_json("./reddit/subreddits/subreddits_2024-01.zst")

Amount of subreddits processed: 16680905it [28:54, 9615.19it/s] 


### Custom display options to enable us to see all columns of subreddits_2025_01.zst

In [15]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 50)

### Display subreddits_2025_01.zst

Function for loading dataframes and cleaning them

In [ ]:
def load_dataframe_from_csv(number,year):
        path = "./data/subreddits"
        df = pd.read_csv(os.path.join(path, f"{year}_dataframe_{number}.csv"))
        print(f"Loaded {year}_dataframe_{number}.csv with shape: {df.shape}")
        df=df.dropna(subset=['description'])
        df=df[['title','description','display_name','url']]
        print(f'Dataframe_{number}.csv after dropping rows with NaN in description and selecting columns: {df.shape}')

        return df


Now we load all files into one big csv file

In [18]:
# dataframes=[]
# for i in range(22):
#     df = load_dataframe_from_csv(i)
#     dataframes.append(df)

big_df_24 = pd.concat(dataframes, ignore_index=True)

In [19]:
# size of the big dataframe
print(f"Big dataframe shape: {big_df_24.shape}")

Big dataframe shape: (610369, 4)


In [20]:
# save the big dataframe to csv
big_df_24.to_csv("./data/subreddits/big_dataframe_24.csv", index=False)

In [46]:
big_df.loc[big_df["display_name"] == "leagueoflegends"]['description'].values[0]

'1. **[Patch 25.S1.3 Bug Megathread](https://www.reddit.com/r/leagueoflegends/comments/1ii1zb4/patch_25s13_bug_megathread/)**\n\n\n[](https://discord.gg/lol)\n\n> [](https://lolesports.com/en-GB#promo) \n\n\n\n### Announcements\n[**Patch 25.S1.3 Notes**](https://www.leagueoflegends.com/en-gb/news/game-updates/patch-25-s1-3-notes/)\n\n[**Patch 25.S1.3 Bug Megathread**](https://www.reddit.com/r/leagueoflegends/comments/1ii1zb4/patch_25s13_bug_megathread/)\n\n\n\n\n\n\n\n###Subreddit Information\n\n[**Read subreddit rules**](/r/leagueoflegends/w/subredditrules#rules)\n\n[**Subreddit Discord**](http://discord.gg/lol)\n\n[**Spoiler free Subreddit**](https://goo.gl/F8Zkt6) \n\n###### START AUTO-GENERATION, DO NOT TOUCH\n#### Upcoming Matches\n\nMatch|Date\n:--|--:\n**LEC**|\nPlayoffs - FNC vs. TH|[21h](https://itsalmo.st/match-113476054450558917)\nPlayoffs - G2 vs. GX|[23h](https://itsalmo.st/match-113476054450558921)\n\nMatch|Date\n:--|--:\n**LTA North**|\n\nMatch|Date\n:--|--:\n**LTA South

In [25]:
big_df_25=pd.read_csv("data/subreddits/big_dataframe.csv")
# size of the big dataframe
print(f"Big dataframe shape: {big_df_25.shape}")

Big dataframe shape: (908790, 4)


Now we combine tge 25 dataset with the 24 dataset

In [26]:
# find union of the two big dataframes
big_df = pd.concat([big_df_24, big_df_25]).drop_duplicates(subset=['display_name'], keep='first').reset_index(drop=True)
print(f"Union of big_df_24 and big_df_25 shape: {big_df.shape}")

Union of big_df_24 and big_df_25 shape: (925958, 4)


### Test for subreddits

Load in the SNAP dataset and create set of subreddits to find all subreddits which are in both datasets

In [27]:
SNAP = pd.read_csv('data/redditHyperlinks-column-filtered.csv')
SNAP_subreddits=set(SNAP['SOURCE_SUBREDDIT'].tolist()+SNAP['TARGET_SUBREDDIT'].tolist())
print('SNAP dataset contains', len(SNAP_subreddits), 'unique subreddits')

SNAP dataset contains 35776 unique subreddits


Now we can create a dataframe of all the subreddits in both the SNAP dataset and the subreddits dataset. We check whether matching on the display name or the url is best:

In [30]:
display_df = big_df[big_df['display_name'].isin(SNAP_subreddits)]
print(f"Filtered dataframe on display name shape: {display_df.shape}")
url_df_both = big_df[big_df['url'].str[3:-1].isin(SNAP_subreddits)]
print(f"Filtered dataframe on url shape: {url_df_both.shape}")

Filtered dataframe on display name shape: (11237, 4)
Filtered dataframe on url shape: (11239, 4)


In [31]:
url_df_25 = big_df_25[big_df_25['url'].str[3:-1].isin(SNAP_subreddits)]
print(f"Filtered dataframe on url shape: {url_df_25.shape}")

Filtered dataframe on url shape: (11119, 4)


In [32]:
# subreddits in the 24 dataframe but not in the 25 dataframe
combined = pd.concat([url_df_25, url_df_both], ignore_index=True)
unique_rows = combined.drop_duplicates(keep=False)
unique_rows

,title,description,display_name,url
7,100 Thieves,>\n\n\n[Read the Rules](https://reddit.com/r/1...,100thieves,/r/100thieves/
9,Things that happened exactly 100 years ago,---\n\n# Rules\n\n1. All submissions must deal...,100yearsago,/r/100yearsago/
13,For fans of the 10mm Auto,___\n**Community guidelines**\n\n* No conducti...,10mm,/r/10mm/
36,Old School RuneScape!,>>>[Submit a link](/r/2007scape/submit#su)\n>>...,2007scape,/r/2007scape/
66,Brevard county and surrounding areas,######[Moving to Brevard? Start here](https://...,321,/r/321/
...,...,...,...,...
22136,Redditvision National Finals,A subreddit for the national finals of /r/redd...,redditvision_nf,/r/redditvision_nf/
22149,The Reddit IMDB video game board,####[Recent activity](https://old.reddit.com/r...,imdbvg,/r/imdbvg/
22188,YouTube TV,##### ◈ [Rules](https://www.reddit.com/r/youtu...,youtubetv,/r/youtubetv/
22205,Nerd City,Nerd City Youtube: [https://www.youtube.com/c/...,nerdcity,/r/nerdcity/


We find the subreddits that are not duplicated across these dataframes to check the difference

In [29]:
combined = pd.concat([display_df, url_df], ignore_index=True)
unique_rows = combined.drop_duplicates(keep=False)
unique_rows

,title,description,display_name,url
22446,redditaholes,a place where all the assholes of reddit are s...,RedditAHoles,/r/redditaholes/
22459,testingforme123,for me,Testingforme123,/r/testingforme123/


It is seen that the difference originates from the capital letters in the display name, and it is chosen to use the dataframe which is mathched on the url. This means that we now have 11119 subreddits to work with. 

In [78]:
url_df.to_csv("./data/COMBINED.csv", index=False)

In [1]:
import pandas as pd
df=pd.read_csv("./data/COMBINED.csv")
df

,title,description,display_name,url,subreddit_name
0,0b0t.org - Minecraft Anarchy Server,Info: 0b0t is a casual minecraft anarchy serve...,0b0t,/r/0b0t/,0b0t
1,reddit moderator statistics and more,###[0bservat0ry.com](http://0bservat0ry.com/re...,0bservat0ry,/r/0bservat0ry/,0bservat0ry
2,0x10c,Subreddit for Notch's cancelled 0x10^c game– a...,0x10c,/r/0x10c/,0x10c
3,vvv,Follow me on Twitter for updates to this chall...,100daysofrejection,/r/100daysofrejection/,100daysofrejection
4,The 100 Movies 365 Days Challenge!,######[Home](http://www.reddit.com#top) [hot](...,100movies365days,/r/100movies365days/,100movies365days
...,...,...,...,...,...
11114,"Zumba: Ditch the workout, join the party!",Zumba Fitness® is the only Latin-inspired danc...,zumba,/r/zumba/,zumba
11115,Zürich,The subreddit for the beautiful city and canto...,zurich,/r/zurich/,zurich
11116,2. Bundesliga,&nbsp;\n\n#2. LIGA TABELLE\n\nPl.|Verein|Sp.|D...,zweiteliga,/r/zweiteliga/,zweiteliga
11117,ZX Spectrum... Colour and Sound...,[General](http://www.reddit.com/r/RetroGamingN...,zxspectrum,/r/zxspectrum/,zxspectrum
